# 0630 Benchmark 문제 풀이

## 1. 라이브러리와 데이터 로드

In [15]:
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:
from huggingface_hub import hf_hub_download
import shutil

repo_id = "hongdune/BenchMarkDataset"
repo_type = "dataset"

files = [
    "train.jsonl",
    "test.jsonl",
    "submission.csv"
]

save_dir = "/content"

for filename in files:
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type=repo_type
    )

    shutil.copy(path, f"{save_dir}/{filename}")
    print(f"saved: {save_dir}/{filename}")

train.jsonl:   0%|          | 0.00/6.85M [00:00<?, ?B/s]

saved: /content/train.jsonl


test.jsonl:   0%|          | 0.00/718k [00:00<?, ?B/s]

saved: /content/test.jsonl


submission.csv:   0%|          | 0.00/9.21k [00:00<?, ?B/s]

saved: /content/submission.csv


In [24]:
train_df = pd.read_json("train.jsonl", lines=True)
train_df.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,row_id
0,A1K582XYLTKUCD,B0042F1L4S,"Coder10 ""Ricardo""","[0, 1]",im not a profesional player but i like to play...,5,Everything a hobbiest want,1361404800,"02 21, 2013",8492
1,A2J4UAF6RW13WK,B000EELB8W,Michael W DeSilva,"[0, 0]",This is just an excellent product. I have been...,5,Excellent Product,1371686400,"06 20, 2013",4666
2,A3OXHLG6DIBRW8,B000BU5V58,"C. Hill ""CFH""","[1, 1]","We opted for the World Tour ""Guitar Gig Bag"" o...",4,Roomy Guitar Case - Recommended,1290556800,"11 24, 2010",4286
3,A3872Y2XH0YDX1,B000CZ0RLK,Amazon Customer,"[0, 0]","This is not a top tier condenser mic, but it i...",5,definitely a good value,1311552000,"07 25, 2011",4446
4,A2G3VQU2GRN8BU,B000978D58,"Moral Hazard ""D""","[1, 3]",This could be used for lightweight microphones...,3,Too cheap.,1316044800,"09 15, 2011",3921


In [49]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9234 entries, 0 to 9233
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   reviewerID        9234 non-null   object
 1   asin              9234 non-null   object
 2   reviewerName      9210 non-null   object
 3   helpful           9234 non-null   object
 4   reviewText        9234 non-null   object
 5   overall           9234 non-null   int64 
 6   summary           9234 non-null   object
 7   unixReviewTime    9234 non-null   int64 
 8   reviewTime        9234 non-null   object
 9   row_id            9234 non-null   int64 
 10  helpful_yes       9234 non-null   int64 
 11  helpful_total     9234 non-null   int64 
 12  len               9234 non-null   int64 
 13  summary_len       9234 non-null   int64 
 14  reviewText_token  9234 non-null   object
 15  summary_token     9234 non-null   object
dtypes: int64(7), object(9)
memory usage: 1.1+ MB


In [50]:
train_df['unixReviewTime'].unique()

array([1361404800, 1371686400, 1290556800, ..., 1307145600, 1296000000,
       1247788800])

In [25]:
# df["helpful"] 컬럼에 들어 있는 값을 분석해서, 그 안에 있는 도움됨 수와 전체 평가 수를 각각 새로운 컬럼으로 분리
import ast

def parse_helpful(x):
    if isinstance(x, list) and len(x) == 2:
        return x[0], x[1]
    if isinstance(x, str):
        y = ast.literal_eval(x)
        return y[0], y[1]
    return 0, 0

train_df["helpful_yes"], train_df["helpful_total"] = zip(*train_df["helpful"].apply(parse_helpful))
train_df.head()


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,row_id,helpful_yes,helpful_total
0,A1K582XYLTKUCD,B0042F1L4S,"Coder10 ""Ricardo""","[0, 1]",im not a profesional player but i like to play...,5,Everything a hobbiest want,1361404800,"02 21, 2013",8492,0,1
1,A2J4UAF6RW13WK,B000EELB8W,Michael W DeSilva,"[0, 0]",This is just an excellent product. I have been...,5,Excellent Product,1371686400,"06 20, 2013",4666,0,0
2,A3OXHLG6DIBRW8,B000BU5V58,"C. Hill ""CFH""","[1, 1]","We opted for the World Tour ""Guitar Gig Bag"" o...",4,Roomy Guitar Case - Recommended,1290556800,"11 24, 2010",4286,1,1
3,A3872Y2XH0YDX1,B000CZ0RLK,Amazon Customer,"[0, 0]","This is not a top tier condenser mic, but it i...",5,definitely a good value,1311552000,"07 25, 2011",4446,0,0
4,A2G3VQU2GRN8BU,B000978D58,"Moral Hazard ""D""","[1, 3]",This could be used for lightweight microphones...,3,Too cheap.,1316044800,"09 15, 2011",3921,1,3


In [26]:
# 새로운 컬럼 생성
train_df['len'] = train_df['reviewText'].fillna('').apply(lambda x: len(x.split()))
train_df['summary_len'] = train_df['summary'].fillna('').apply(lambda x: len(x.split()))

- helpful
    
    해당 리뷰가 받은 도움 평가로, `[helpful_yes, helpful_total]` 형태의 리스트로 제공됩니다.
    
- overall
    
    사용자가 해당 상품에 대해 부여한 평점입니다.
    
- row_id
    
    각 데이터 샘플을 식별하기 위한 고유 ID입니다.

In [27]:
train_df[["overall", "unixReviewTime", "helpful_yes", "helpful_total", "len", "summary_len"]].corr()

,overall,unixReviewTime,helpful_yes,helpful_total,len,summary_len
overall,1.000000,-0.015340,-0.013419,-0.050759,-0.067820,-0.084525
unixReviewTime,-0.015340,1.000000,-0.309734,-0.323309,-0.180326,-0.047982
helpful_yes,-0.013419,-0.309734,1.000000,0.986931,0.283004,0.083200
helpful_total,-0.050759,-0.323309,0.986931,1.000000,0.291343,0.087880
len,-0.067820,-0.180326,0.283004,0.291343,1.000000,0.283745
summary_len,-0.084525,-0.047982,0.083200,0.087880,0.283745,1.000000


리뷰에 대해 평가한 사람이 많을수록, “도움이 됐다”고 누른 사람 수도 많다.  
긴 리뷰일수록 정보가 많아서 사람들이 더 많이 평가하거나, 도움이 된다고 누를 가능성이 약간 있다.  
평점(overall)은 리뷰 길이, 도움됨 수, 요약문 길이와 거의 관계가 없습니다.  

In [ ]:
print(train_df['len'].describe())
print('\n')
print(train_df['summary_len'].describe())

count    9234.000000
mean       90.338965
std       109.347447
min         0.000000
25%        31.000000
50%        54.000000
75%       105.000000
max      2043.000000
Name: len, dtype: float64


count    9234.000000
mean        4.376760
std         2.857689
min         1.000000
25%         2.000000
50%         4.000000
75%         6.000000
max        25.000000
Name: summary_len, dtype: float64


In [28]:
# 리뷰 없이 요약본만 존재하는 경우
count_zero_len = len(train_df[train_df["len"] == 0])
print(f"길이가 0인 행의 수 = {count_zero_len}\n")
print(train_df[train_df["len"] == 0][["reviewText", "summary", "len", "summary_len"]])

길이가 0인 행의 수 = 6

     reviewText                                        summary  len  \
1820             This would be a must-have at twice the price.    0   
4627                                   Mini tech for musicians    0   
6252                              To make you sound like a pro    0   
7344                 No power = No Sound, But It Sounds GREAT!    0   
7816                   Great sound and features for the price!    0   
8002                                           great foot rest    0   

      summary_len  
1820            9  
4627            4  
6252            7  
7344            9  
7816            7  
8002            3  


In [29]:
# 토크나이저
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
def preprocessing(data, max_length):
    if pd.isna(data):
        data = ""
    result = tokenizer(
        data,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='np'
    )
    return result["input_ids"][0]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [30]:
MAX_LEN_REVIEW = 105
MAX_LEN_SUMMARY = 10

train_df["reviewText_token"] = train_df["reviewText"].apply(lambda x: preprocessing(x, MAX_LEN_REVIEW))
train_df["summary_token"] = train_df["summary"].apply(lambda x: preprocessing(x, MAX_LEN_SUMMARY))
train_df.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,row_id,helpful_yes,helpful_total,len,summary_len,reviewText_token,summary_token
0,A1K582XYLTKUCD,B0042F1L4S,"Coder10 ""Ricardo""","[0, 1]",im not a profesional player but i like to play...,5,Everything a hobbiest want,1361404800,"02 21, 2013",8492,0,1,21,4,"[101, 10047, 2025, 1037, 11268, 2229, 19301, 2...","[101, 2673, 1037, 7570, 27982, 2102, 2215, 102..."
1,A2J4UAF6RW13WK,B000EELB8W,Michael W DeSilva,"[0, 0]",This is just an excellent product. I have been...,5,Excellent Product,1371686400,"06 20, 2013",4666,0,0,151,2,"[101, 2023, 2003, 2074, 2019, 6581, 4031, 1012...","[101, 6581, 4031, 102, 0, 0, 0, 0, 0, 0]"
2,A3OXHLG6DIBRW8,B000BU5V58,"C. Hill ""CFH""","[1, 1]","We opted for the World Tour ""Guitar Gig Bag"" o...",4,Roomy Guitar Case - Recommended,1290556800,"11 24, 2010",4286,1,1,152,5,"[101, 2057, 12132, 2005, 1996, 2088, 2778, 100...","[101, 2282, 2100, 2858, 2553, 1011, 6749, 102,..."
3,A3872Y2XH0YDX1,B000CZ0RLK,Amazon Customer,"[0, 0]","This is not a top tier condenser mic, but it i...",5,definitely a good value,1311552000,"07 25, 2011",4446,0,0,65,4,"[101, 2023, 2003, 2025, 1037, 2327, 7563, 2470...","[101, 5791, 1037, 2204, 3643, 102, 0, 0, 0, 0]"
4,A2G3VQU2GRN8BU,B000978D58,"Moral Hazard ""D""","[1, 3]",This could be used for lightweight microphones...,3,Too cheap.,1316044800,"09 15, 2011",3921,1,3,62,2,"[101, 2023, 2071, 2022, 2109, 2005, 12038, 155...","[101, 2205, 10036, 1012, 102, 0, 0, 0, 0, 0]"


In [34]:
temp = train_df[["reviewText_token", "summary_token", "len", "summary_len", "overall", "helpful_yes", "helpful_total"]]
temp.head()

,reviewText_token,summary_token,len,summary_len,overall,helpful_yes,helpful_total
0,"[101, 10047, 2025, 1037, 11268, 2229, 19301, 2...","[101, 2673, 1037, 7570, 27982, 2102, 2215, 102...",21,4,5,0,1
1,"[101, 2023, 2003, 2074, 2019, 6581, 4031, 1012...","[101, 6581, 4031, 102, 0, 0, 0, 0, 0, 0]",151,2,5,0,0
2,"[101, 2057, 12132, 2005, 1996, 2088, 2778, 100...","[101, 2282, 2100, 2858, 2553, 1011, 6749, 102,...",152,5,4,1,1
3,"[101, 2023, 2003, 2025, 1037, 2327, 7563, 2470...","[101, 5791, 1037, 2204, 3643, 102, 0, 0, 0, 0]",65,4,5,0,0
4,"[101, 2023, 2071, 2022, 2109, 2005, 12038, 155...","[101, 2205, 10036, 1012, 102, 0, 0, 0, 0, 0]",62,2,3,1,3


In [36]:
class ReviewDataset(Dataset):
    def __init__(self, df):
        self.data = df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return {
            'review_tokens': torch.tensor(self.data["reviewText_token"][idx], dtype=torch.long),
            'summary_tokens': torch.tensor(self.data["summary_token"][idx], dtype=torch.long),
            'overall': torch.tensor(self.data["overall"][idx], dtype=torch.float),
            'helpful': torch.tensor(self.data[["helpful_yes","helpful_total"]].iloc[idx].values, dtype=torch.float)
        }

train_dataset = ReviewDataset(train_df)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
for i in train_loader:
    print(i)
    break

{'review_tokens': tensor([[ 101, 2025, 2172,  ..., 2039, 2278,  102],
        [ 101, 1045, 2359,  ..., 1056, 3786,  102],
        [ 101, 2157, 2041,  ...,    0,    0,    0],
        ...,
        [ 101, 2023, 2003,  ..., 1037, 9587,  102],
        [ 101, 1045, 1005,  ...,    0,    0,    0],
        [ 101, 2028, 1997,  ...,    0,    0,    0]]), 'summary_tokens': tensor([[  101,  2515,  1996,  7577,   102,     0,     0,     0,     0,     0],
        [  101, 15749,  2604,  2804,   102,     0,     0,     0,     0,     0],
        [  101,  6581,  3643,   999,   102,     0,     0,     0,     0,     0],
        [  101,  3019,  9391,  1010,  2021, 19120,  2003,  2673,   102,     0],
        [  101,  2034,  3751,  2858,   102,     0,     0,     0,     0,     0],
        [  101,  2204,  6218,  2302,  2151,  7540, 13704,   102,     0,     0],
        [  101,  2515,  2025,  4906,  4202,  2858,   102,     0,     0,     0],
        [  101,  4832,  2041,   102,     0,     0,     0,     0,     0,     0

In [37]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_review = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.rnn_summary = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.overall = nn.Linear(hidden_dim * 2, 1)
        self.helpful = nn.Linear(hidden_dim * 2, 2)

    def forward(self, review_tokens, summary_tokens):
        review_embed = self.embedding(review_tokens)
        # print(review_embed.size())
        summary_embed = self.embedding(summary_tokens)
        # print(summary_embed.size())
        _, review_hidden = self.rnn_review(review_embed)
        _, summary_hidden = self.rnn_summary(summary_embed)

        review_hidden = review_hidden[0].squeeze(0)  # (batch_size, hidden_dim)
        # print(review_hidden.size())
        summary_hidden = summary_hidden[0].squeeze(0)  # (batch_size, hidden_dim)
        # print(summary_hidden.size())
        combined_hidden = torch.cat([review_hidden, summary_hidden], dim=1)
        # print(combined_hidden.size())
        overall_pred = self.overall(combined_hidden)
        helpful_pred = self.helpful(combined_hidden)

        return overall_pred, helpful_pred

In [38]:
vocab_size = 30522
embed_dim = 128
hidden_dim = 64

model = RNN(vocab_size, embed_dim, hidden_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [46]:
print(train_df['helpful_total'].unique())
print(train_df['helpful_yes'].unique())

[  1   0   3   9   2  28   8   7   4  10  12   5   6  13  22  14  37  15
  16  72  30  34  11  20  19  76 161  61  25  27  65  31  77  40  44  24
  18 160  33  17  53  35  57  49 130  21 101  47  26 300  67 114  89  50
  52  29  63 117  48  23 136 266  42  82  39  51  32 192  75 150  41 259
 166  70  43  64  45  36  93  59  38  81  68 182 193]
[  0   1   3   2   7  27   6   4  10  11   5   9  12  22   8  36  13  15
  70  18  17  74 157  54  25  26  14  61  72  40  44  21 156  38  16  23
  52  32  24  35  49 114  20  47 290  63  31  97  81  48  28 112  30  46
 130 259  19  88  77  34 190  73 142 246 161  65  42  41  93  33  29  58
 189  75  67  45  53 174 188]


In [39]:
model.to(device)

def train(model, train_loader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for batch in train_loader:
            review = batch["review_tokens"].to(device)
            summary = batch["summary_tokens"].to(device)
            overall = batch["overall"].to(device)
            helpful = batch["helpful"].to(device)

            optimizer.zero_grad()

            overall_pred, likes_pred = model(review, summary)
            # print(overall_pred.size(), likes_pred.size())
            # print(overall.size())
            loss_overall = criterion(overall_pred, overall.view(-1, 1))
            loss_likes = criterion(likes_pred, helpful)
            loss = loss_overall + loss_likes
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

train(model, train_loader, criterion, optimizer, epochs=10)

Epoch 1, Loss: 84.19558895087985
Epoch 2, Loss: 82.794928893086
Epoch 3, Loss: 78.24429319133809
Epoch 4, Loss: 68.20906556869461
Epoch 5, Loss: 57.20458224240471
Epoch 6, Loss: 48.706393982867056
Epoch 7, Loss: 42.627378032230176
Epoch 8, Loss: 38.6382215163287
Epoch 9, Loss: 34.27429318706469
Epoch 10, Loss: 30.074059419256592


In [40]:
test_df = pd.read_json("test.jsonl", lines=True)

if "row_id" not in test_df.columns:
    test_df = test_df.reset_index(drop=True)
    test_df["row_id"] = test_df.index.astype(int)

test_df["reviewText"] = test_df["reviewText"].fillna("")
test_df["summary"] = test_df["summary"].fillna("")

test_df["review_tokens"] = test_df["reviewText"].apply(
    lambda x: tokenizer.encode(
        x,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LEN_REVIEW,
    )
)

test_df["summary_tokens"] = test_df["summary"].apply(
    lambda x: tokenizer.encode(
        x,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LEN_SUMMARY,
    )
)

In [41]:
class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        return {
            "row_id": int(r["row_id"]),
            "review_tokens": torch.tensor(r["review_tokens"], dtype=torch.long),
            "summary_tokens": torch.tensor(r["summary_tokens"], dtype=torch.long),
        }

def collate_fn(batch, pad_id=0):
    row_ids = torch.tensor([b["row_id"] for b in batch])

    reviews = [b["review_tokens"] for b in batch]
    summaries = [b["summary_tokens"] for b in batch]

    reviews = torch.nn.utils.rnn.pad_sequence(
        reviews, batch_first=True, padding_value=pad_id
    )
    summaries = torch.nn.utils.rnn.pad_sequence(
        summaries, batch_first=True, padding_value=pad_id
    )

    return {"row_id": row_ids, "review_tokens": reviews, "summary_tokens": summaries}

test_ds = TestDataset(test_df)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [42]:
import numpy as np

model.eval()

# row_id 예측값 매핑 (DataFrame 대신 dict로 보관)
pred_map = {}

with torch.no_grad():
    for batch in test_loader:
        row_ids = batch["row_id"].tolist()
        review  = batch["review_tokens"].to(device)
        summary = batch["summary_tokens"].to(device)

        overall_pred, helpful_pred = model(review, summary)
        overall_pred = overall_pred.squeeze(-1).cpu().numpy()
        helpful_pred = helpful_pred.cpu().numpy()

        for rid, o, hp in zip(row_ids, overall_pred, helpful_pred):
            pred_map[int(rid)] = {
                "helpful_yes_pred":   float(hp[0]),
                "helpful_total_pred": float(hp[1]),
                "overall_pred":       float(o),
            }

In [43]:
SUBMISSION_PATH = "submission.csv"

submission = pd.read_csv(SUBMISSION_PATH)

# row_id 기준으로 예측값 채워넣기
submission["helpful_yes_pred"]   = submission["row_id"].map(lambda r: pred_map.get(int(r), {}).get("helpful_yes_pred"))
submission["helpful_total_pred"] = submission["row_id"].map(lambda r: pred_map.get(int(r), {}).get("helpful_total_pred"))
submission["overall_pred"]       = submission["row_id"].map(lambda r: pred_map.get(int(r), {}).get("overall_pred"))

# 누락 체크
missing = submission[submission[["helpful_yes_pred","helpful_total_pred","overall_pred"]].isna().any(axis=1)]
if len(missing) > 0:
    print(f"[경고] 예측이 채워지지 않은 row 수: {len(missing)}")
    print(missing.head())

submission.to_csv(SUBMISSION_PATH, index=False)
print(submission.head())
print("shape:", submission.shape)

   row_id  helpful_yes_pred  helpful_total_pred  overall_pred
0    7056          0.754497            0.955065      4.316884
1    6141          9.361698           10.505390      4.050445
2    5873          0.910961            1.235201      4.081289
3    4806          0.869216            0.951203      4.467285
4     919          0.275893            0.333765      4.580671
shape: (1027, 4)
